# ANRF AISEHack 2.0 — Polymer Property Prediction (FULL run)

Standalone Kaggle notebook. Predicts **Tg** (glass transition, °C) and **Egc** (chain band gap, eV)
from polymer SMILES; scored on **mean R²** across both.

**Pipeline:** RDKit features (2D descriptors + ECFP4 + ECFP6 + RDKitFP + MACCS) → **LGBM + XGB** ensemble,
plus an **AttentiveFP GNN** on the molecular graph, combined with **honest nested blend weights**
(weights fit out-of-fold, so the reported CV number generalizes).

**Full settings:** 5 seeds × 10 folds trees (4000 estimators, early-stopped), GNN 3 seeds × 5 folds.
Everything seeded (SEED=42). Enable the **GPU accelerator** so the GNN uses CUDA.
Expect a few hours of runtime on Kaggle (well within the 9-hour limit); the trees dominate.


In [1]:
# Kaggle has torch preinstalled (with CUDA on GPU kernels); add rdkit + torch_geometric.
!pip install -q rdkit torch_geometric

import glob, gc, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb

from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator

import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn.models import AttentiveFP

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# On Kaggle GPU, CUDA handles variable-shape GNN batches well, so prefer it.
# (On Apple MPS the same GNN is slower due to per-shape kernel recompilation, but that
#  doesn't apply here.)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch device:", DEVICE)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 66.5 MB/s eta 0:00:00
Torch device: cuda


In [2]:
# ---- FULL-run configuration ----
SEEDS        = [42, 7, 123, 17, 99]  # tree seeds
N_FOLDS      = 10                    # tree CV folds
N_ESTIMATORS = 4000                  # trees (early-stopped)
GNN_SEEDS    = [42, 7, 123]          # GNN seeds
GNN_FOLDS    = 5                     # GNN CV folds
TARGETS      = ("tg", "egc")

In [3]:
# ---- load competition data & split by target_type ----
train_path = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]
test_path  = glob.glob("/kaggle/input/**/test.csv",  recursive=True)[0]
train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

def split_by_type(name):
    tr = train[train["target_type"] == name].reset_index(drop=True)
    te = test[test["target_type"] == name].reset_index(drop=True)
    return tr, te, tr["target"].values.astype(float)

for name in TARGETS:
    tr, te, y = split_by_type(name)
    print(f"{name:>3}  train {len(tr):>5}  test {len(te):>5}  y-range [{y.min():.3f}, {y.max():.3f}]")

 tg  train  4143  test  2763  y-range [-118.000, 490.000]
egc  train  2028  test  1352  y-range [0.103, 9.863]


In [4]:
# ---- RDKit featurization + per-target preprocessor ----
DESC_NAMES  = [n for n, _ in Descriptors.descList]
MORGAN_ECFP4 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
MORGAN_ECFP6 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
RDKIT_FPGEN  = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048)

def featurize(smiles_list):
    rdkit_rows, ecfp4_rows, ecfp6_rows, rdk_rows, maccs_rows = [], [], [], [], []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rdkit_rows.append([np.nan] * len(DESC_NAMES))
            ecfp4_rows.append(np.zeros(2048, dtype=np.uint8))
            ecfp6_rows.append(np.zeros(2048, dtype=np.uint8))
            rdk_rows.append(np.zeros(2048, dtype=np.uint8))
            maccs_rows.append(np.zeros(167, dtype=np.uint8))
        else:
            rdkit_rows.append(list(Descriptors.CalcMolDescriptors(mol).values()))
            ecfp4_rows.append(MORGAN_ECFP4.GetFingerprintAsNumPy(mol))
            ecfp6_rows.append(MORGAN_ECFP6.GetFingerprintAsNumPy(mol))
            rdk_rows.append(RDKIT_FPGEN.GetFingerprintAsNumPy(mol))
            maccs_rows.append(np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.uint8))
    return pd.concat([
        pd.DataFrame(rdkit_rows, columns=DESC_NAMES),
        pd.DataFrame(ecfp4_rows, columns=[f"ecfp4_{i}" for i in range(2048)]),
        pd.DataFrame(ecfp6_rows, columns=[f"ecfp6_{i}" for i in range(2048)]),
        pd.DataFrame(rdk_rows,   columns=[f"rdkfp_{i}" for i in range(2048)]),
        pd.DataFrame(maccs_rows, columns=[f"maccs_{i}" for i in range(167)]),
    ], axis=1)

def build_preprocessor(X_raw):
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.dropna(axis=1, thresh=int(0.2 * len(X)))
    X = X.loc[:, X.var() > 0]
    good = X.columns.tolist()
    imp = SimpleImputer(strategy="median")
    sc  = StandardScaler()
    Xs  = sc.fit_transform(imp.fit_transform(X))
    return Xs, (imp, sc, good)

def apply_preprocessor(X_raw, prep):
    imp, sc, good = prep
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.reindex(columns=good, fill_value=np.nan)
    return sc.transform(imp.transform(X))

In [5]:
# ---- LGBM + XGB seed x fold ensemble ----
def lgbm_params(t, n_estimators):
    p = dict(objective="regression", metric="rmse", n_estimators=n_estimators,
             learning_rate=0.01, num_leaves=127, max_depth=-1, min_child_samples=15,
             subsample=0.8, subsample_freq=1, colsample_bytree=0.4,
             reg_alpha=0.05, reg_lambda=1.0, n_jobs=-1, verbose=-1)
    if t == "egc": p["num_leaves"], p["min_child_samples"] = 63, 20
    return p

def xgb_params(t, n_estimators):
    p = dict(objective="reg:squarederror", n_estimators=n_estimators, learning_rate=0.01,
             max_depth=6, min_child_weight=5, subsample=0.8, colsample_bytree=0.4,
             reg_alpha=0.05, reg_lambda=1.0, n_jobs=-1, tree_method="hist",
             early_stopping_rounds=200)
    if t == "egc": p["max_depth"] = 5
    return p

def train_trees(Xtr, y, Xte, t, seeds, n_splits, n_estimators):
    Xtr, y, Xte = np.asarray(Xtr, np.float32), np.asarray(y, float), np.asarray(Xte, np.float32)
    oof_all, test_all = [], []
    for seed in seeds:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        ol, ox = np.zeros(len(Xtr)), np.zeros(len(Xtr))
        tl, tx = np.zeros(len(Xte)), np.zeros(len(Xte))
        lp = lgbm_params(t, n_estimators); lp["random_state"] = seed
        xp = xgb_params(t, n_estimators);  xp["random_state"] = seed
        for tr, va in kf.split(Xtr):
            ml = lgb.LGBMRegressor(**lp)
            ml.fit(Xtr[tr], y[tr], eval_set=[(Xtr[va], y[va])],
                   callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])
            ol[va] = ml.predict(Xtr[va]); tl += ml.predict(Xte) / n_splits
            mx = xgb.XGBRegressor(**xp)
            mx.fit(Xtr[tr], y[tr], eval_set=[(Xtr[va], y[va])], verbose=False)
            ox[va] = mx.predict(Xtr[va]); tx += mx.predict(Xte) / n_splits
        oof_all.append((ol, ox)); test_all.append((tl, tx))
        print(f"    {t} seed {seed} | OOF LGBM={r2_score(y, ol):.4f}  XGB={r2_score(y, ox):.4f}")
    oof  = {"lgbm": np.mean([o[0] for o in oof_all], 0), "xgb": np.mean([o[1] for o in oof_all], 0)}
    test = {"lgbm": np.mean([o[0] for o in test_all], 0), "xgb": np.mean([o[1] for o in test_all], 0)}
    return oof, test

In [6]:
# ---- AttentiveFP graph neural network ----
ATOM_LIST = ["C","N","O","S","F","Si","P","Cl","Br","I","B","*","Other"]
NODE_DIM, EDGE_DIM = len(ATOM_LIST) + 5, 6

def _oh(v, ch):
    x = [0]*len(ch); x[ch.index(v) if v in ch else len(ch)-1] = 1; return x

def atom_features(a):
    return _oh(a.GetSymbol(), ATOM_LIST) + [a.GetDegree(), a.GetFormalCharge(),
            int(a.GetIsAromatic()), a.GetTotalNumHs(), int(a.GetHybridization())]

def bond_features(b):
    bt = b.GetBondType()
    return [int(bt==Chem.rdchem.BondType.SINGLE), int(bt==Chem.rdchem.BondType.DOUBLE),
            int(bt==Chem.rdchem.BondType.TRIPLE), int(bt==Chem.rdchem.BondType.AROMATIC),
            int(b.GetIsConjugated()), int(b.IsInRing())]

def smiles_to_graph(smi, y=None):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_features(b)
        ei += [[i, j], [j, i]]; ea += [bf, bf]
    if not ei: ei, ea = [[0, 0]], [[0]*EDGE_DIM]
    d = Data(x=x, edge_index=torch.tensor(ei, dtype=torch.long).t().contiguous(),
             edge_attr=torch.tensor(ea, dtype=torch.float))
    if y is not None: d.y = torch.tensor([y], dtype=torch.float)
    return d

def _gnn_one_seed(graphs, y_raw, gtest, scaler, seed, n_splits,
                  epochs=80, patience=15, lr=2e-3, batch_size=512):
    torch.manual_seed(seed); np.random.seed(seed)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof, test = np.zeros(len(graphs)), np.zeros(len(gtest))
    test_loader = DataLoader(gtest, batch_size=batch_size*2, shuffle=False)
    for fold, (tr, va) in enumerate(kf.split(np.arange(len(graphs))), 1):
        tl = DataLoader([graphs[i] for i in tr], batch_size=batch_size, shuffle=True)
        vl = DataLoader([graphs[i] for i in va], batch_size=batch_size*2, shuffle=False)
        m = AttentiveFP(in_channels=NODE_DIM, hidden_channels=64, out_channels=1,
                        edge_dim=EDGE_DIM, num_layers=2, num_timesteps=2, dropout=0.1).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-5); lf = nn.MSELoss()
        best, bad, state, last = float("inf"), 0, None, 0
        for ep in range(epochs):
            last = ep; m.train()
            for b in tl:
                b = b.to(DEVICE); opt.zero_grad()
                loss = lf(m(b.x, b.edge_index, b.edge_attr, b.batch).squeeze(-1), b.y)
                loss.backward(); opt.step()
            m.eval(); vloss = []
            with torch.no_grad():
                for b in vl:
                    b = b.to(DEVICE)
                    vloss.append(lf(m(b.x, b.edge_index, b.edge_attr, b.batch).squeeze(-1), b.y).item())
            v = float(np.mean(vloss))
            if v < best:
                best, bad = v, 0
                state = {k: val.detach().cpu().clone() for k, val in m.state_dict().items()}
            else:
                bad += 1
                if bad >= patience: break
        m.load_state_dict(state); m.eval()
        with torch.no_grad():
            vp = []
            for b in vl:
                b = b.to(DEVICE); vp.append(m(b.x, b.edge_index, b.edge_attr, b.batch).squeeze(-1).cpu().numpy())
            oof[va] = np.concatenate(vp)
            tp = []
            for b in test_loader:
                b = b.to(DEVICE); tp.append(m(b.x, b.edge_index, b.edge_attr, b.batch).squeeze(-1).cpu().numpy())
            test += np.concatenate(tp) / n_splits
        r2 = r2_score(y_raw[va], scaler.inverse_transform(oof[va].reshape(-1,1)).ravel())
        print(f"    fold {fold} | epochs={last+1:2d}  GNN R²={r2:.4f}")
        del m, opt, state
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
        gc.collect()
    return (scaler.inverse_transform(oof.reshape(-1,1)).ravel(),
            scaler.inverse_transform(test.reshape(-1,1)).ravel())

def train_gnn(tr_smiles, y, te_smiles, t, seeds, n_splits):
    scaler = StandardScaler()
    ys = scaler.fit_transform(np.asarray(y, float).reshape(-1,1)).ravel()
    graphs = [smiles_to_graph(s, v) for s, v in zip(tr_smiles, ys)]
    gtest  = [smiles_to_graph(s) for s in te_smiles]
    oof_all, test_all = [], []
    for seed in seeds:
        print(f"  {t} GNN seed={seed}")
        o, te = _gnn_one_seed(graphs, np.asarray(y, float), gtest, scaler, seed, n_splits)
        oof_all.append(o); test_all.append(te)
    return {"gnn": np.mean(oof_all, 0)}, {"gnn": np.mean(test_all, 0)}

In [7]:
# ---- honest nested blend weights ----
def _fit_w(P, y):
    n = P.shape[1]
    r = minimize(lambda w: -r2_score(y, P @ w), np.full(n, 1/n), method="SLSQP",
                 bounds=[(0,1)]*n, constraints={"type":"eq","fun":lambda w: w.sum()-1})
    return r.x

def honest_cv_r2(oof, y, n_folds=5, seed=SEED):
    P = np.column_stack([oof[m] for m in oof]); y = np.asarray(y, float)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed); pred = np.zeros(len(y))
    for tr, va in kf.split(P):
        pred[va] = P[va] @ _fit_w(P[tr], y[tr])
    return r2_score(y, pred)

def final_weights(oof, y):
    P = np.column_stack([oof[m] for m in oof])
    return dict(zip(oof, _fit_w(P, np.asarray(y, float))))

def error_corr(oof, y):
    y = np.asarray(y, float); e = {m: oof[m]-y for m in oof}; names = list(oof); out = {}
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            out[(names[i], names[j])] = float(np.corrcoef(e[names[i]], e[names[j]])[0,1])
    return out

In [8]:
# ---- run: trees + GNN -> honest blend -> submission ----
parts, honest_by_target = [], {}
for name in TARGETS:
    print("="*60, f"\n  {name.upper()}\n", "="*60)
    tr, te, y = split_by_type(name)

    Xtr_raw = featurize(tr["smiles"].tolist())
    Xte_raw = featurize(te["smiles"].tolist())
    Xtr, prep = build_preprocessor(Xtr_raw)
    Xte = apply_preprocessor(Xte_raw, prep)
    print("  features:", Xtr.shape[1])

    oof_t, testp_t = train_trees(Xtr, y, Xte, name, SEEDS, N_FOLDS, N_ESTIMATORS)
    oof_g, testp_g = train_gnn(tr["smiles"].tolist(), y, te["smiles"].tolist(), name, GNN_SEEDS, GNN_FOLDS)
    oof   = {**oof_t, **oof_g}
    testp = {**testp_t, **testp_g}   # per-model test predictions (do NOT shadow `test`)

    per = {m: r2_score(y, oof[m]) for m in oof}
    corr = error_corr(oof, y)
    honest = honest_cv_r2(oof, y)
    w = final_weights(oof, y)
    test_blend = sum(w[m]*testp[m] for m in w)

    print("  per-model OOF R²:", {m: round(v,4) for m,v in per.items()})
    print("  error corr:", {f"{a}~{b}": round(c,3) for (a,b),c in corr.items()})
    print("  weights:", {m: round(v,3) for m,v in w.items()})
    print(f"  HONEST held-out mean-blend R² = {honest:.4f}")
    honest_by_target[name] = honest

    sub = te[["id"]].copy(); sub["target"] = test_blend; parts.append(sub)

submission = pd.concat(parts, axis=0).sort_values("id").reset_index(drop=True)
assert submission["target"].isna().sum() == 0
submission.to_csv("submission.csv", index=False)

mean_honest = np.mean(list(honest_by_target.values()))
print("\n" + "="*60)
print("  honest R² by target:", {k: round(v,4) for k,v in honest_by_target.items()})
print(f"  MEAN HONEST R² = {mean_honest:.4f}")
print("  submission rows:", len(submission))
print("="*60)
print(submission.head().to_string())

  TG
  features: 6442
    tg seed 42 | OOF LGBM=0.9028  XGB=0.9052
    tg seed 7 | OOF LGBM=0.9033  XGB=0.9060
    tg seed 123 | OOF LGBM=0.9037  XGB=0.9063
    tg seed 17 | OOF LGBM=0.9042  XGB=0.9064
    tg seed 99 | OOF LGBM=0.9035  XGB=0.9058
  tg GNN seed=42
    fold 1 | epochs=80  GNN R²=0.8064
    fold 2 | epochs=80  GNN R²=0.8546
    fold 3 | epochs=80  GNN R²=0.8379
    fold 4 | epochs=80  GNN R²=0.8229
    fold 5 | epochs=80  GNN R²=0.8320
  tg GNN seed=7
    fold 1 | epochs=80  GNN R²=0.8201
    fold 2 | epochs=80  GNN R²=0.8398
    fold 3 | epochs=80  GNN R²=0.8231
    fold 4 | epochs=78  GNN R²=0.8240
    fold 5 | epochs=80  GNN R²=0.8210
  tg GNN seed=123
    fold 1 | epochs=80  GNN R²=0.8208
    fold 2 | epochs=80  GNN R²=0.8150
    fold 3 | epochs=80  GNN R²=0.8510
    fold 4 | epochs=80  GNN R²=0.8345
    fold 5 | epochs=80  GNN R²=0.8109
  per-model OOF R²: {'lgbm': 0.9058, 'xgb': 0.9082, 'gnn': 0.8342}
  error corr: {'lgbm~xgb': 0.985, 'lgbm~gnn': 0.751, 'xgb~gnn': 0